<a href="https://colab.research.google.com/github/SolisProcopioUriel/SimulacionII/blob/main/Estimacion_parametros_medicamentos_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Uriel Solis Procopio

In [17]:
!pip install plotly

#Librerias

In [18]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.special import expit
from scipy.optimize import differential_evolution
from scipy.interpolate import interp1d, CubicSpline
import time
import matplotlib.pyplot as plt

#Parámetros fijos estimados con AG

In [19]:
parametros_fijos = {
    'q1': 0.4250068,
    'q2': 0.0049551,
    'q3': 0.4494950,
    'g': 1.2341919,
    'Cmax': 5.9154248,
    'Cmin': 2.2445529,
    'Smax': 3.9460791,
    'd': 0.2838233,
    'V': 1.2352702,
}

#Sistema de ecuaciones diferenciales

Consideremos a $M(t)$ como "medicamento" y la expasión en serie de Taylor de segundo orden centrada en $t_0$ queda de la siguiente manera

  \begin{equation*}
    M(t) = a_0 + a_1(t-t_0)+a_2(t-t_0)^2
  \end{equation*}

  en la ecuación diferencial es $-l\cdot M(t) \cdot T$

In [ ]:
def M(t, a0, a1, a2, t0):
    return a0 + a1 * (t - t0) + a2 * (t - t0)**2

def sistema_EDO(volumen, nutrientes, tiempo, l, a0, a1, a2, t0):
    volumen = np.clip(volumen, 0, 2.0)
    nutrientes = np.clip(nutrientes, 0, 10.0)

    sigmoide = expit(parametros_fijos['Cmin'] - nutrientes)

    medicamento = M(tiempo, a0, a1, a2, t0)

    dV = (
        parametros_fijos['q2'] * nutrientes * volumen * (parametros_fijos['Smax'] - (volumen + parametros_fijos['V']))
        - parametros_fijos['d'] * (parametros_fijos['Cmin'] - nutrientes) * sigmoide * volumen
        - l * medicamento * volumen
    )

    dC = (
        parametros_fijos['g'] * (parametros_fijos['Cmax'] - nutrientes) * parametros_fijos['V']
        - parametros_fijos['q1'] * nutrientes * volumen
        - parametros_fijos['q3'] * nutrientes * volumen * (parametros_fijos['Smax'] - (volumen + parametros_fijos['V']))
    )

    return np.clip([dV, dC], -5, 5)

#Runge-Kutta de orden 4

In [ ]:
def runge_kutta_4(vol_inicial, nut_inicial, tiempo_final, paso, l, a0, a1, a2, t0):
    tiempos = np.arange(0, tiempo_final + paso, paso)
    vol = np.zeros_like(tiempos)
    nut = np.zeros_like(tiempos)
    vol[0], nut[0] = vol_inicial, nut_inicial

    for i in range(len(tiempos) - 1):
        t = tiempos[i]
        V, C = vol[i], nut[i]
        if not np.isfinite(V) or not np.isfinite(C):
            vol[i + 1:] = np.nan
            break

        k1 = paso * sistema_EDO(V, C, t, l, a0, a1, a2, t0)
        k2 = paso * sistema_EDO(V + 0.5 * k1[0], C + 0.5 * k1[1], t + 0.5*paso, l, a0, a1, a2, t0)
        k3 = paso * sistema_EDO(V + 0.5 * k2[0], C + 0.5 * k2[1], t + 0.5*paso, l, a0, a1, a2, t0)
        k4 = paso * sistema_EDO(V + k3[0], C + k3[1], t + paso, l, a0, a1, a2, t0)

        vol[i + 1] = V + (k1[0] + 2*k2[0] + 2*k3[0] + k4[0]) / 6
        nut[i + 1] = C + (k1[1] + 2*k2[1] + 2*k3[1] + k4[1]) / 6

    return tiempos, vol

#Función objetivo

In [ ]:
def crear_funcion_objetivo(vol_inicial, nut_inicial, tiempos_datos, datos_reales):
    def funcion(params):
        l, a0, a1, a2, t0 = params
        try:
            t, V = runge_kutta_4(vol_inicial, nut_inicial, max(tiempos_datos)+1, 0.05, l, a0, a1, a2, t0)
            if np.any(np.isnan(V)):
                return 1e6
            interpolador = interp1d(t, V, kind='linear', fill_value="extrapolate")
            predicciones = interpolador(tiempos_datos)
            return np.mean(np.abs(predicciones - datos_reales))  # Error medio absoluto
        except:
            return 1e6
    return funcion

#Función principal de estimación y visualización

In [ ]:
def ajustar_y_graficar_modelo(datos_reales, tiempos_datos, nombre_titulo=""):
    vol_inicial = float(datos_reales[0])
    nut_inicial = 2.5

    #inicio = time.time()

    funcion_objetivo = crear_funcion_objetivo(vol_inicial, nut_inicial, tiempos_datos, datos_reales)
    limites = [(0, 1.0), (0, 2.0), (-0.01, 0.2), (-0.01, 0.01), (5, 20)]
    resultado = differential_evolution(funcion_objetivo, limites, maxiter=300, popsize=30, seed=42)
    l, a0, a1, a2, t0 = resultado.x

    print("\nParámetros óptimos encontrados:")
    for nombre, val in zip(["l", "a0", "a1", "a2", "t0"], resultado.x):
        print(f"{nombre} = {val:.6f}")

    # Simulación con parámetros óptimos
    t_sim, V_sim = runge_kutta_4(vol_inicial, nut_inicial, max(tiempos_datos) + 1, 0.01, l, a0, a1, a2, t0)
    spline = CubicSpline(t_sim, V_sim)
    V_interpolado = spline(tiempos_datos)

    tabla = pd.DataFrame({
        "Tiempo (días)": tiempos_datos,
        "Volumen real (cm³)": datos_reales,
        "Modelo T(t) (cm³)": V_interpolado
    })
    print("\nTabla comparativa:")
    print(tabla.to_string(index=False, float_format="%.4f"))

    # Gráfica interactiva con Plotly
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_sim, y=V_sim, mode='lines', name='Modelo ajustado', line=dict(color='purple')))
    fig.add_trace(go.Scatter(x=tiempos_datos, y=datos_reales, mode='markers+text',
                             name='Datos reales', marker=dict(color='red', size=10),
                             text=[f"{val:.3f}" for val in datos_reales],
                             textposition="top center"))

    fig.update_layout(title=f"Modelo de crecimiento tumoral {nombre_titulo}",
                      xaxis_title='Tiempo (días)',
                      yaxis_title='Volumen tumoral [cm³]',
                      width=800, height=500)
    fig.show()

    #fin = time.time()
    #print(f"Tiempo total de ejecución: {fin - inicio:.2f} segundos")

#Grafica de la función $M(t)$ expandido en serie de Taylor

In [20]:
def graficar_M(a0, a1, a2, t0, t_min=0, t_max=27, num_puntos=300):
    """
    Grafica la función M(t) = a0 + a1*(t - t0) + a2*(t - t0)^2 con los parámetros dados.

    Parámetros:
    - a0, a1, a2: Coeficientes de la función M(t)
    - t0: Punto de referencia en el tiempo (por defecto 11.205653)
    - t_min, t_max: Intervalo de t a graficar
    - num_puntos: Resolución de la gráfica
    """

    # Definir la función M(t)
    def M(t):
        return a0 + a1 * (t - t0) + a2 * (t - t0)**2

    # Valores de t y de M(t)
    t = np.linspace(t_min, t_max, num_puntos)
    m = M(t)

    # Crear figura
    fig = go.Figure()

    # Traza principal
    fig.add_trace(go.Scatter(
        x=t, y=m, mode='lines',
        name=fr'$M(t) = {a0:.3f} + {a1:.3f}(t - {t0:.2f}) + {a2:.3f}(t - {t0:.2f})^2$',
        line=dict(color='orange')
    ))

    # Líneas guía
    fig.add_trace(go.Scatter(
        x=[t_min, t_max], y=[0, 0],
        mode='lines', line=dict(color='black', width=1),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=[0, 0], y=[min(m), max(m)],
        mode='lines', line=dict(color='black', width=1),
        showlegend=False
    ))

    # Personalización del diseño
    fig.update_layout(
        title='Función $M(t)$ con parámetros personalizados',
        xaxis=dict(
            title='t',
            tickmode='linear',
            dtick=1
        ),
        yaxis=dict(
            title='M(t)',
            tickmode='linear',
            dtick=1
        ),
        legend=dict(x=0.01, y=0.99),
        template='simple_white'
    )

    fig.show()

#Docetaxcel

In [ ]:
datos1 = np.array([0.116, 0.179, 0.222, 0.244, 0.252, 0.247, 0.237, 0.257, 0.273])
tiempos1 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos1, tiempos1, "Docetaxcel - Ratón J000101173")


Parámetros óptimos encontrados:
l = 0.613136
a0 = 1.628512
a1 = 0.113296
a2 = -0.009545
t0 = 11.205653

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1160             0.1160
             3              0.1790             0.1623
             6              0.2220             0.2080
            10              0.2440             0.2442
            13              0.2520             0.2520
            17              0.2470             0.2470
            20              0.2370             0.2417
            24              0.2570             0.2474
            27              0.2730             0.2731


In [33]:
graficar_M(a0=1.628512, a1=0.113296, a2=--0.009545, t0=11.205653)

In [ ]:
datos2 = np.array([0.131, 0.172, 0.21, 0.237, 0.229, 0.242, 0.196, 0.17, 0.169])
tiempos2 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos2, tiempos2, "Docetaxcel - Ratón TM01117")


Parámetros óptimos encontrados:
l = 0.762743
a0 = 0.756901
a1 = 0.146608
a2 = -0.005066
t0 = 5.054104

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1310             0.1310
             3              0.1720             0.1761
             6              0.2100             0.2100
            10              0.2370             0.2311
            13              0.2290             0.2297
            17              0.2420             0.2128
            20              0.1960             0.1960
            24              0.1700             0.1767
            27              0.1690             0.1689


In [34]:
graficar_M(a0=0.756901, a1=0.146608, a2=-0.005066, t0=5.054104)

In [ ]:
datos2 = np.array([0.139, 0.143, 0.158, 0.163, 0.152, 0.128, 0.072, 0.056, 0.047])
tiempos2 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos2, tiempos2, "Docetaxcel - Ratón J000100675")


Parámetros óptimos encontrados:
l = 0.056068
a0 = 1.952126
a1 = 0.100502
a2 = 0.006160
t0 = 8.939034

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1390             0.1390
             3              0.1430             0.1500
             6              0.1580             0.1579
            10              0.1630             0.1603
            13              0.1520             0.1521
            17              0.1280             0.1254
            20              0.0720             0.0964
            24              0.0560             0.0560
            27              0.0470             0.0315


In [ ]:
datos2 = np.array([0.118, 0.125, 0.131, 0.136, 0.131, 0.121, 0.11, 0.108, 0.142])
tiempos2 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos2, tiempos2, "Docetaxcel - Ratón J000102184")


Parámetros óptimos encontrados:
l = 0.071738
a0 = 1.759198
a1 = -0.002964
a2 = -0.005684
t0 = 13.488546

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1180             0.1180
             3              0.1250             0.1339
             6              0.1310             0.1393
            10              0.1360             0.1358
            13              0.1310             0.1291
            17              0.1210             0.1208
            20              0.1100             0.1183
            24              0.1080             0.1247
            27              0.1420             0.1420


In [ ]:
datos10 = np.array([0.117, 0.198, 0.247, 0.271, 0.336, 0.396, 0.475, 0.532, 0.806])
tiempos10 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos10, tiempos10, "Docetaxcel - Ratón J000103634")


Parámetros óptimos encontrados:
l = 0.054220
a0 = 1.402615
a1 = -0.008014
a2 = -0.009933
t0 = 15.338198

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1170             0.1170
             3              0.1980             0.1747
             6              0.2470             0.2307
            10              0.2710             0.2961
            13              0.3360             0.3381
            17              0.3960             0.3959
            20              0.4750             0.4559
            24              0.5320             0.5982
            27              0.8060             0.8060


In [ ]:
datos11 = np.array([0.136, 0.207, 0.161, 0.156, 0.075, 0.042, 0.054, 0.026, 0.025])
tiempos11 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos11, tiempos11, "Docetaxcel - Ratón J000103917")


Parámetros óptimos encontrados:
l = 0.285472
a0 = 0.894970
a1 = 0.035050
a2 = -0.005751
t0 = 11.141785

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1360             0.1360
             3              0.2070             0.1877
             6              0.1610             0.1846
            10              0.1560             0.1255
            13              0.0750             0.0800
            17              0.0420             0.0420
            20              0.0540             0.0282
            24              0.0260             0.0219
            27              0.0250             0.0251


In [ ]:
datos12 = np.array([0.146, 0.29, 0.351, 0.341, 0.427, 0.437, 0.55, 0.701, 0.852])
tiempos12 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos12, tiempos12, "Docetaxcel - Ratón TM00091")


Parámetros óptimos encontrados:
l = 0.050968
a0 = 0.769725
a1 = 0.180369
a2 = -0.009640
t0 = 6.641067

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1460             0.1460
             3              0.2900             0.2184
             6              0.3510             0.2899
            10              0.3410             0.3741
            13              0.4270             0.4263
            17              0.4370             0.4909
            20              0.5500             0.5500
            24              0.7010             0.6782
            27              0.8520             0.8524


In [ ]:
datos13 = np.array([0.115, 0.126, 0.156, 0.155, 0.171, 0.18, 0.181, 0.178, 0.198])
tiempos13 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos13, tiempos13, "Docetaxcel - Ratón TM00103")


Parámetros óptimos encontrados:
l = 0.067521
a0 = 1.154653
a1 = 0.058018
a2 = -0.002340
t0 = 5.679765

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1150             0.1150
             3              0.1260             0.1349
             6              0.1560             0.1499
            10              0.1550             0.1641
            13              0.1710             0.1710
            17              0.1800             0.1771
            20              0.1810             0.1810
            24              0.1780             0.1885
            27              0.1980             0.1980


In [ ]:
datos14 = np.array([0.102, 0.117, 0.141, 0.176, 0.18, 0.204, 0.224, 0.239, 0.229])
tiempos14 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos14, tiempos14, "Docetaxcel - Ratón TM00999")


Parámetros óptimos encontrados:
l = 0.051066
a0 = 1.269102
a1 = 0.020687
a2 = 0.002782
t0 = 9.496936

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1020             0.1020
             3              0.1170             0.1171
             6              0.1410             0.1338
            10              0.1760             0.1598
            13              0.1800             0.1810
            17              0.2040             0.2079
            20              0.2240             0.2239
            24              0.2390             0.2337
            27              0.2290             0.2290


In [ ]:
datos15 = np.array([0.116, 0.128, 0.123, 0.102, 0.089, 0.068, 0.059, 0.049, 0.046])
tiempos15 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos15, tiempos15, "Docetaxcel - Ratón TM01079")


Parámetros óptimos encontrados:
l = 0.113294
a0 = 1.315974
a1 = 0.055849
a2 = -0.004053
t0 = 8.409626

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1160             0.1160
             3              0.1280             0.1264
             6              0.1230             0.1230
            10              0.1020             0.1055
            13              0.0890             0.0890
            17              0.0680             0.0690
            20              0.0590             0.0577
            24              0.0490             0.0484
            27              0.0460             0.0460


In [ ]:
datos16 = np.array([0.14, 0.217, 0.225, 0.35, 0.476, 0.586, 0.594, 0.695, 0.698])
tiempos16 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos16, tiempos16, "Docetaxcel - Ratón TM01278")


Parámetros óptimos encontrados:
l = 0.032167
a0 = 1.002036
a1 = 0.170079
a2 = -0.001127
t0 = 8.773350

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1400             0.1400
             3              0.2170             0.2020
             6              0.2250             0.2748
            10              0.3500             0.3871
            13              0.4760             0.4760
            17              0.5860             0.5854
            20              0.5940             0.6487
            24              0.6950             0.6950
            27              0.6980             0.6980


#Doxorrubicina

In [ ]:
datos1 = np.array([0.131, 0.186, 0.219, 0.260, 0.323, 0.357, 0.354, 0.379, 0.377])
tiempos1 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos1, tiempos1, "Doxorrubicina - Ratón TM01117")


Parámetros óptimos encontrados:
l = 0.724250
a0 = 1.288367
a1 = 0.049650
a2 = -0.001254
t0 = 16.977670

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1310             0.1310
             3              0.1860             0.1750
             6              0.2190             0.2194
            10              0.2600             0.2766
            13              0.3230             0.3139
            17              0.3570             0.3518
            20              0.3540             0.3696
            24              0.3790             0.3791
            27              0.3770             0.3770


In [ ]:
datos5 = np.array([0.138, 0.151, 0.172, 0.223, 0.308, 0.391, 0.523, 0.561, 0.674])
tiempos5 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos5, tiempos5, "Doxorrubicina - Ratón J000100675")


Parámetros óptimos encontrados:
l = 0.542782
a0 = 0.601592
a1 = 0.019729
a2 = 0.006797
t0 = 14.942507

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1380             0.1380
             3              0.1510             0.1496
             6              0.1720             0.1721
            10              0.2230             0.2243
            13              0.3080             0.2833
            17              0.3910             0.3910
            20              0.5230             0.4888
            24              0.5610             0.6151
            27              0.6740             0.6740


In [ ]:
datos21 = np.array([0.118, 0.166, 0.188, 0.184, 0.2, 0.229, 0.241, 0.349, 0.377])
tiempos21 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos21, tiempos21, "Doxorrubicina - Ratón J000101173")


Parámetros óptimos encontrados:
l = 0.047070
a0 = 1.542137
a1 = 0.068583
a2 = -0.005288
t0 = 6.486405

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1180             0.1180
             3              0.1660             0.1421
             6              0.1880             0.1618
            10              0.1840             0.1843
            13              0.2000             0.2001
            17              0.2290             0.2245
            20              0.2410             0.2501
            24              0.3490             0.3054
            27              0.3770             0.3770


In [ ]:
datos22 = np.array([0.108, 0.132, 0.157, 0.189, 0.23, 0.263, 0.343, 0.427, 0.489])
tiempos22 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos22, tiempos22, "Doxorrubicina - Ratón J000102184")


Parámetros óptimos encontrados:
l = 0.028864
a0 = 1.908213
a1 = 0.033947
a2 = 0.001906
t0 = 16.447085

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1080             0.1080
             3              0.1320             0.1289
             6              0.1570             0.1528
            10              0.1890             0.1928
            13              0.2300             0.2299
            17              0.2630             0.2899
            20              0.3430             0.3430
            24              0.4270             0.4237
            27              0.4890             0.4889


In [ ]:
datos23 = np.array([0.117, 0.17, 0.223, 0.322, 0.421, 0.436, 0.542, 0.653, 0.901])
tiempos23 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos23, tiempos23, "Doxorrubicina - Ratón J000103634")


Parámetros óptimos encontrados:
l = 0.069647
a0 = 0.804073
a1 = 0.096923
a2 = -0.008403
t0 = 9.366413

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1170             0.1170
             3              0.1700             0.1813
             6              0.2230             0.2456
            10              0.3220             0.3218
            13              0.4210             0.3706
            17              0.4360             0.4365
            20              0.5420             0.5040
            24              0.6530             0.6650
            27              0.9010             0.9010


In [ ]:
datos23 = np.array([0.132, 0.222, 0.304, 0.359, 0.494, 0.572, 0.701, 0.897, 1.035])
tiempos23 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos23, tiempos23, "Doxorrubicina - Ratón J000103917")


Parámetros óptimos encontrados:
l = 0.033814
a0 = 0.977800
a1 = 0.195561
a2 = -0.009943
t0 = 8.531896

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1320             0.1320
             3              0.2220             0.2026
             6              0.3040             0.2820
            10              0.3590             0.3959
            13              0.4940             0.4832
            17              0.5720             0.6027
            20              0.7010             0.7012
            24              0.8970             0.8649
            27              1.0350             1.0350


In [ ]:
datos24 = np.array([0.138, 0.23, 0.268, 0.375, 0.366, 0.464, 0.541, 0.46, 0.623])
tiempos24 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos24, tiempos24, "Doxorrubicina - Ratón TM00091")


Parámetros óptimos encontrados:
l = 0.066990
a0 = 1.354469
a1 = 0.047915
a2 = -0.006600
t0 = 14.752941

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1380             0.1380
             3              0.2300             0.2116
             6              0.2680             0.2858
            10              0.3750             0.3713
            13              0.3660             0.4188
            17              0.4640             0.4635
            20              0.5410             0.4928
            24              0.4600             0.5480
            27              0.6230             0.6229


In [ ]:
datos25 = np.array([0.115, 0.149, 0.146, 0.186, 0.195, 0.232, 0.267, 0.256, 0.279])
tiempos25 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos25, tiempos25, "Doxorrubicina - Ratón TM00103")


Parámetros óptimos encontrados:
l = 0.057769
a0 = 1.336834
a1 = 0.032892
a2 = -0.000211
t0 = 12.755931

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1150             0.1150
             3              0.1490             0.1367
             6              0.1460             0.1576
            10              0.1860             0.1858
            13              0.1950             0.2064
            17              0.2320             0.2321
            20              0.2670             0.2492
            24              0.2560             0.2682
            27              0.2790             0.2790


In [ ]:
datos26 = np.array([0.102, 0.152, 0.178, 0.211, 0.25, 0.322, 0.338, 0.415, 0.433])
tiempos26 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos26, tiempos26, "Doxorrubicina - Ratón TM00999")


Parámetros óptimos encontrados:
l = 0.038798
a0 = 1.211460
a1 = 0.059793
a2 = 0.002021
t0 = 10.753881

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1020             0.1020
             3              0.1520             0.1294
             6              0.1780             0.1608
            10              0.2110             0.2111
            13              0.2500             0.2543
            17              0.3220             0.3159
            20              0.3380             0.3610
            24              0.4150             0.4107
            27              0.4330             0.4331


In [ ]:
datos27 = np.array([0.114, 0.162, 0.222, 0.289, 0.369, 0.469, 0.576, 0.698, 0.786])
tiempos27 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos27, tiempos27, "Doxorrubicina - Ratón TM01079")


Parámetros óptimos encontrados:
l = 0.043249
a0 = 0.622112
a1 = 0.079155
a2 = -0.000932
t0 = 7.977824

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1140             0.1140
             3              0.1620             0.1581
             6              0.2220             0.2100
            10              0.2890             0.2943
            13              0.3690             0.3689
            17              0.4690             0.4819
            20              0.5760             0.5737
            24              0.6980             0.6980
            27              0.7860             0.7860


In [21]:
datos28 = np.array([0.125, 0.223, 0.335, 0.401, 0.507, 0.623, 0.839, 0.819, 0.995])
tiempos28 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos28, tiempos28, "Doxorrubicina - Ratón TM01278")


Parámetros óptimos encontrados:
l = 0.042302
a0 = 1.703524
a1 = 0.000832
a2 = -0.009005
t0 = 19.103606

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1250             0.1250
             3              0.2230             0.2016
             6              0.3350             0.2897
            10              0.4010             0.4151
            13              0.5070             0.5070
            17              0.6230             0.6230
            20              0.8390             0.7108
            24              0.8190             0.8501
            27              0.9950             0.9952


#Cisplatino

In [22]:
datos3 = np.array([0.131, 0.197, 0.289, 0.376, 0.496, 0.601, 0.704, 0.822, 0.999])
tiempos3 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos3, tiempos3, "Cisplatino - Ratón TM01117")


Parámetros óptimos encontrados:
l = 0.052268
a0 = 1.191371
a1 = 0.067760
a2 = -0.007292
t0 = 13.673291

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1310             0.1310
             3              0.1970             0.2056
             6              0.2890             0.2890
            10              0.3760             0.4054
            13              0.4960             0.4905
            17              0.6010             0.6010
            20              0.7040             0.6893
            24              0.8220             0.8380
            27              0.9990             0.9990


In [ ]:
datos6 = np.array([0.139, 0.144, 0.186, 0.206, 0.275, 0.31, 0.359, 0.389, 0.516])
tiempos6 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos6, tiempos6, "Cisplatino - Ratón J000100675")


Parámetros óptimos encontrados:
l = 0.348813
a0 = 1.872276
a1 = 0.001367
a2 = -0.000562
t0 = 14.225880

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1390             0.1390
             3              0.1440             0.1619
             6              0.1860             0.1859
            10              0.2060             0.2233
            13              0.2750             0.2566
            17              0.3100             0.3100
            20              0.3590             0.3590
            24              0.3890             0.4401
            27              0.5160             0.5160


In [23]:
datos31 = np.array([0.122, 0.171, 0.21, 0.244, 0.291, 0.326, 0.349, 0.389, 0.444])
tiempos31 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos31, tiempos31, "Cisplatino - Ratón J000101173")


Parámetros óptimos encontrados:
l = 0.048922
a0 = 1.831883
a1 = 0.012705
a2 = -0.006007
t0 = 17.301140

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1220             0.1220
             3              0.1710             0.1677
             6              0.2100             0.2107
            10              0.2440             0.2605
            13              0.2910             0.2907
            17              0.3260             0.3242
            20              0.3490             0.3490
            24              0.3890             0.3925
            27              0.4440             0.4440


In [24]:
datos31 = np.array([0.109, 0.126, 0.15, 0.148, 0.166, 0.178, 0.193, 0.214, 0.297])
tiempos31 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos31, tiempos31, "Cisplatino - Ratón J000102184")


Parámetros óptimos encontrados:
l = 0.053380
a0 = 1.678968
a1 = 0.053755
a2 = -0.006875
t0 = 8.590494

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1090             0.1090
             3              0.1260             0.1310
             6              0.1500             0.1462
            10              0.1480             0.1591
            13              0.1660             0.1660
            17              0.1780             0.1775
            20              0.1930             0.1930
            24              0.2140             0.2344
            27              0.2970             0.2970


In [25]:
datos32 = np.array([0.119, 0.209, 0.331, 0.361, 0.636, 0.736, 0.925, 1.072, 1.419])
tiempos32 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos32, tiempos32, "Cisplatino - Ratón J000103634")


Parámetros óptimos encontrados:
l = 0.070156
a0 = 0.946777
a1 = 0.007248
a2 = -0.009617
t0 = 16.120592

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1190             0.1190
             3              0.2090             0.2157
             6              0.3310             0.3309
            10              0.3610             0.4901
            13              0.6360             0.5999
            17              0.7360             0.7367
            20              0.9250             0.8546
            24              1.0720             1.0962
            27              1.4190             1.4190


In [26]:
datos33 = np.array([0.145, 0.19, 0.251, 0.225, 0.198, 0.234, 0.277, 0.331, 0.37])
tiempos33 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos33, tiempos33, "Cisplatino - Ratón J000103917")


Parámetros óptimos encontrados:
l = 0.058025
a0 = 1.601108
a1 = 0.010479
a2 = -0.004459
t0 = 13.499925

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1450             0.1450
             3              0.1900             0.1759
             6              0.2510             0.2003
            10              0.2250             0.2250
            13              0.1980             0.2394
            17              0.2340             0.2582
            20              0.2770             0.2767
            24              0.3310             0.3171
            27              0.3700             0.3700


In [28]:
datos34 = np.array([0.143, 0.231, 0.346, 0.466, 0.656, 0.763, 0.955, 0.841, 1.153])
tiempos34 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos34, tiempos34, "Cisplatino - Ratón TM00091")


Parámetros óptimos encontrados:
l = 0.055110
a0 = 1.101090
a1 = 0.077892
a2 = -0.007070
t0 = 13.992732

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1430             0.1430
             3              0.2310             0.2360
             6              0.3460             0.3461
            10              0.4660             0.5060
            13              0.6560             0.6227
            17              0.7630             0.7634
            20              0.9550             0.8618
            24              0.8410             1.0069
            27              1.1530             1.1525


In [29]:
datos35 = np.array([0.118, 0.142, 0.16, 0.176, 0.205, 0.237, 0.252, 0.272, 0.297])
tiempos35 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos35, tiempos35, "Cisplatino - Ratón TM00103")


Parámetros óptimos encontrados:
l = 0.073714
a0 = 1.052787
a1 = 0.020584
a2 = -0.000570
t0 = 12.427419

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1180             0.1180
             3              0.1420             0.1396
             6              0.1600             0.1599
            10              0.1760             0.1870
            13              0.2050             0.2069
            17              0.2370             0.2329
            20              0.2520             0.2520
            24              0.2720             0.2774
            27              0.2970             0.2970


In [30]:
datos36 = np.array([0.102, 0.145, 0.174, 0.191, 0.217, 0.192, 0.166, 0.133, 0.124])
tiempos36 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos36, tiempos36, "Cisplatino - Ratón TM00999")


Parámetros óptimos encontrados:
l = 0.084200
a0 = 0.772408
a1 = 0.123638
a2 = -0.003705
t0 = 6.716459

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1020             0.1020
             3              0.1450             0.1418
             6              0.1740             0.1740
            10              0.1910             0.1979
            13              0.2170             0.1995
            17              0.1920             0.1846
            20              0.1660             0.1665
            24              0.1330             0.1407
            27              0.1240             0.1240


In [31]:
datos37 = np.array([0.118, 0.138, 0.129, 0.118, 0.112, 0.075, 0.064, 0.05, 0.03])
tiempos37 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos37, tiempos37, "Cisplatino - Ratón TM01079")


Parámetros óptimos encontrados:
l = 0.079723
a0 = 1.869158
a1 = 0.073565
a2 = -0.000840
t0 = 11.374368

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1180             0.1180
             3              0.1380             0.1281
             6              0.1290             0.1290
            10              0.1180             0.1188
            13              0.1120             0.1047
            17              0.0750             0.0816
            20              0.0640             0.0640
            24              0.0500             0.0433
            27              0.0300             0.0309


In [32]:
datos38 = np.array([0.139, 0.222, 0.263, 0.362, 0.515, 0.625, 0.737, 0.982, 1.063])
tiempos38 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos38, tiempos38, "Cisplatino - Ratón TM01278")


Parámetros óptimos encontrados:
l = 0.034100
a0 = 1.520364
a1 = 0.055783
a2 = -0.002580
t0 = 17.526998

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1390             0.1390
             3              0.2220             0.1956
             6              0.2630             0.2628
            10              0.3620             0.3731
            13              0.5150             0.4721
            17              0.6250             0.6249
            20              0.7370             0.7518
            24              0.9820             0.9292
            27              1.0630             1.0630


#Ciclofosfamida

In [ ]:
datos4 = np.array([0.129, 0.199, 0.253, 0.262, 0.362, 0.457, 0.5, 0.542, 0.65])
tiempos4 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos4, tiempos4, "Ciclofosfamida - Ratón TM01117")


Parámetros óptimos encontrados:
l = 0.278187
a0 = 1.177153
a1 = 0.185203
a2 = -0.005089
t0 = 6.198479

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1290             0.1290
             3              0.1990             0.1776
             6              0.2530             0.2301
            10              0.2620             0.3051
            13              0.3620             0.3630
            17              0.4570             0.4408
            20              0.5000             0.4998
            24              0.5420             0.5819
            27              0.6500             0.6501


In [35]:
datos40 = np.array([0.138, 0.145, 0.184, 0.222, 0.302, 0.407, 0.51, 0.567, 0.828])
tiempos40 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos40, tiempos40, "Ciclofosfamida - Ratón J000100675")


Parámetros óptimos encontrados:
l = 0.047728
a0 = 0.766727
a1 = -0.003638
a2 = 0.003183
t0 = 15.297338

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1380             0.1380
             3              0.1450             0.1576
             6              0.1840             0.1846
            10              0.2220             0.2384
            13              0.3020             0.2962
            17              0.4070             0.4034
            20              0.5100             0.5102
            24              0.5670             0.6852
            27              0.8280             0.8277


In [36]:
datos41 = np.array([0.117, 0.159, 0.215, 0.289, 0.361, 0.434, 0.464, 0.487, 0.547])
tiempos41 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos41, tiempos41, "Ciclofosfamida - Ratón J000101173")


Parámetros óptimos encontrados:
l = 0.040342
a0 = 0.880310
a1 = 0.128827
a2 = -0.002400
t0 = 7.623971

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1170             0.1170
             3              0.1590             0.1638
             6              0.2150             0.2152
            10              0.2890             0.2895
            13              0.3610             0.3460
            17              0.4340             0.4170
            20              0.4640             0.4640
            24              0.4870             0.5160
            27              0.5470             0.5470


In [37]:
datos42 = np.array([0.108, 0.122, 0.157, 0.176, 0.222, 0.27, 0.338, 0.413, 0.531])
tiempos42 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos42, tiempos42, "Ciclofosfamida - Ratón J000102184")


Parámetros óptimos encontrados:
l = 0.065964
a0 = 0.891778
a1 = 0.023718
a2 = -0.002428
t0 = 7.632458

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1080             0.1080
             3              0.1220             0.1327
             6              0.1570             0.1569
            10              0.1760             0.1916
            13              0.2220             0.2211
            17              0.2700             0.2700
            20              0.3380             0.3195
            24              0.4130             0.4169
            27              0.5310             0.5310


In [38]:
datos43 = np.array([0.118, 0.184, 0.274, 0.387, 0.524, 0.647, 0.916, 1.13, 1.463])
tiempos43 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos43, tiempos43, "Ciclofosfamida - Ratón J000103634")


Parámetros óptimos encontrados:
l = 0.075586
a0 = 0.478722
a1 = 0.053843
a2 = -0.005531
t0 = 11.344673

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1180             0.1180
             3              0.1840             0.1890
             6              0.2740             0.2725
            10              0.3870             0.4005
            13              0.5240             0.5083
            17              0.6470             0.6762
            20              0.9160             0.8348
            24              1.1300             1.1299
            27              1.4630             1.4630


In [39]:
datos44 = np.array([0.13, 0.207, 0.312, 0.403, 0.566, 0.702, 0.853, 1.125, 1.334])
tiempos44 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos44, tiempos44, "Ciclofosfamida - Ratón J000103917")


Parámetros óptimos encontrados:
l = 0.053815
a0 = 0.824423
a1 = 0.066744
a2 = -0.006698
t0 = 12.713678

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1300             0.1300
             3              0.2070             0.2064
             6              0.3120             0.2965
            10              0.4030             0.4342
            13              0.5660             0.5472
            17              0.7020             0.7117
            20              0.8530             0.8529
            24              1.1250             1.0896
            27              1.3340             1.3340


In [40]:
datos45 = np.array([0.139, 0.212, 0.214, 0.285, 0.411, 0.425, 0.519, 0.436, 0.565])
tiempos45 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos45, tiempos45, "Ciclofosfamida - Ratón TM00091")


Parámetros óptimos encontrados:
l = 0.054655
a0 = 1.244692
a1 = 0.062539
a2 = 0.001419
t0 = 16.871768

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1390             0.1390
             3              0.2120             0.1759
             6              0.2140             0.2184
            10              0.2850             0.2861
            13              0.4110             0.3441
            17              0.4250             0.4255
            20              0.5190             0.4832
            24              0.4360             0.5427
            27              0.5650             0.5650


In [41]:
datos46 = np.array([0.115, 0.137, 0.159, 0.184, 0.212, 0.254, 0.287, 0.326, 0.346])
tiempos46 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos46, tiempos46, "Ciclofosfamida - Ratón TM00103")


Parámetros óptimos encontrados:
l = 0.060873
a0 = 1.016528
a1 = 0.012936
a2 = 0.001824
t0 = 11.694409

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1150             0.1150
             3              0.1370             0.1322
             6              0.1590             0.1517
            10              0.1840             0.1840
            13              0.2120             0.2125
            17              0.2540             0.2547
            20              0.2870             0.2870
            24              0.3260             0.3255
            27              0.3460             0.3460


In [42]:
datos47 = np.array([0.101, 0.124, 0.145, 0.148, 0.194, 0.199, 0.244, 0.253, 0.221])
tiempos47 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos47, tiempos47, "Ciclofosfamida - Ratón TM00999")


Parámetros óptimos encontrados:
l = 0.084893
a0 = 0.785666
a1 = 0.042243
a2 = 0.003191
t0 = 14.099096

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1010             0.1010
             3              0.1240             0.1162
             6              0.1450             0.1351
            10              0.1480             0.1670
            13              0.1940             0.1940
            17              0.1990             0.2275
            20              0.2440             0.2440
            24              0.2530             0.2435
            27              0.2210             0.2216


In [43]:
datos48 = np.array([0.119, 0.161, 0.189, 0.202, 0.208, 0.209, 0.206, 0.196, 0.184])
tiempos48 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos48, tiempos48, "Ciclofosfamida - Ratón TM01079")


Parámetros óptimos encontrados:
l = 0.073237
a0 = 1.498082
a1 = 0.047740
a2 = -0.002732
t0 = 14.440151

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1190             0.1190
             3              0.1610             0.1518
             6              0.1890             0.1784
            10              0.2020             0.2020
            13              0.2080             0.2099
            17              0.2090             0.2093
            20              0.2060             0.2031
            24              0.1960             0.1919
            27              0.1840             0.1841


In [44]:
datos49 = np.array([0.128, 0.199, 0.272, 0.378, 0.476, 0.516, 0.62, 0.758, 0.897])
tiempos49 = np.array([0, 3, 6, 10, 13, 17, 20, 24, 27])
ajustar_y_graficar_modelo(datos49, tiempos49, "Ciclofosfamida - Ratón TM01278")


Parámetros óptimos encontrados:
l = 0.068521
a0 = 0.800505
a1 = 0.078831
a2 = -0.005622
t0 = 11.126650

Tabla comparativa:
 Tiempo (días)  Volumen real (cm³)  Modelo T(t) (cm³)
             0              0.1280             0.1280
             3              0.1990             0.1981
             6              0.2720             0.2746
            10              0.3780             0.3780
            13              0.4760             0.4513
            17              0.5160             0.5448
            20              0.6200             0.6199
            24              0.7580             0.7504
            27              0.8970             0.8970
